In [ ]:
from eduray import *
from eduray.internals import * #only for visualizer it to look at the scene setup, not needed for animation itself

## Animation setup example: Smooth camera movement around a transparent sphere
> **Note** This ray-tracer is not primarily designed for animation, so the rendering times can be quite long, especially with higher quality settings. For this reason, the example below uses extra low settings to keep the rendering time manageable.

In [ ]:
camera = PinholeCamera(
    origin=Vertex(0, 0.5, 5.0),
    direction=Vector(0.3, -0.05, -1),
    fov_deg=40.0,
)

# Key light (warm, upper right) + fill light (cool, upper left)
key_light = PointLight(
    position=Vertex(5, 6, 5),
    intensity=1.2,
    color=Color.custom_rgb(255, 245, 230),
)
fill_light = PointLight(
    position=Vertex(-4, 4, 3),
    intensity=0.5,
    color=Color.custom_rgb(220, 230, 255),
)
ambient_light = AmbientLight(
    intensity=0.15,
    color=Color.custom_rgb(255, 255, 255),
)

# Hero glass sphere — sits on floor (radius 1, center y=0, floor y=-1)
glass_sphere = Object(
    geometry=Sphere(),
    material=PhongMaterial(
        base_color=Color.custom_rgb(240, 240, 255),
        transparency=0.95,
        ior=1.5,
        reflectivity=0.08,
        shininess=120,
    ),
)

# Two small gems flanking it — also resting on the floor (y_center = -0.5)
diamond_sphere = Object(
    geometry=Sphere(center=Vertex(1.8, -0.5, -0.2), radius=0.5),
    material=PhongMaterial(
        base_color=Color.custom_rgb(210, 245, 255),
        spec_color=Color.custom_rgb(255, 255, 255),
        shininess=250,
        ior=2.42,
        reflectivity=0.1,
        transparency=0.92,
    ),
)
crystal_sphere = Object(
    geometry=Sphere(center=Vertex(-1.8, -0.5, -0.2), radius=0.5),
    material=PhongMaterial(
        base_color=Color.custom_rgb(200, 255, 255),
        spec_color=Color.custom_rgb(255, 255, 255),
        shininess=200,
        ior=1.8,
        reflectivity=0.06,
        transparency=0.9,
    ),
)

floor = Object(
    geometry=Plane(
        point=Vertex(0, -1, 0),
        normal=Vector(0, 1, 0),
    ),
    material=PhongMaterial(
        base_color=Color.custom_rgb(220, 220, 220),
        shininess=30,
        reflectivity=0.15,
    ),
)

# Colored spheres arranged in an arc BEHIND the glass for clean refraction
red_sphere = Object(
    geometry=Sphere(center=Vertex(-1.1, -0.4, -3.0), radius=0.6),
    material=PhongMaterial(
        base_color=Color.custom_rgb(220, 60, 60),
        shininess=40,
    ),
)
green_sphere = Object(
    geometry=Sphere(center=Vertex(1.1, -0.4, -3.0), radius=0.6),
    material=PhongMaterial(
        base_color=Color.custom_rgb(60, 200, 80),
        shininess=40,
    ),
)
blue_sphere = Object(
    geometry=Sphere(center=Vertex(0.0, -0.3, -4.2), radius=0.7),
    material=PhongMaterial(
        base_color=Color.custom_rgb(70, 100, 230),
        shininess=40,
    ),
)

objects=[glass_sphere, diamond_sphere, crystal_sphere, floor, red_sphere, green_sphere, blue_sphere]

scene = Scene(
    camera=camera,
    lights=[key_light, fill_light, ambient_light],
    objects=objects,
    skybox="../skyboxes/sand.hdr"
)

### Render configuration

In [ ]:
R64p = Resolution.custom(112, 64),
render_config = RenderConfig(
    resolution=R64p,
    samples_per_pixel=1,
    max_depth=3,
)

rt = LinearRenderLoop(
    render_config = render_config,
    scene = scene
)

### Animation setup

In [ ]:
animation_setup = AnimationSetup(
    # translation
    move_from=Vertex(-2.5, 0.2, 4.2),
    move_to=Vertex(6.5, 0.4, 4.2),
    move_start_delay=0.0,
    move_duration=4.0,

    # I would rework this to look at, which is implemented in the `Camera` class
    rotate_axis=Vector(0, 1, 0),
    rotate_angle_deg=70.0,
    rotate_start_delay=0.0,
    rotate_duration=4.0,

    # slight zoom-in to enhance the sense of movement and focus on the glass sphere
    zoom_from=50.0,
    zoom_to=20.0,
    zoom_start_delay=1.0,
    zoom_duration=3.0,
)

animator = Animator(
    animation_setup=animation_setup,
    animation_fps=5,
    animation_length_seconds=4.0,
    ray_tracer=rt,
)

#### visualise the scene setup
It can be pain to wait for the animation frames to render, so it's a good idea to visualize the scene setup first using the `Visualizer`.

In [ ]:
vis = Visualizer()
vis.create_empty_scene(
    size=4.0,
    figsize=(12, 8),
)
vis.visualize_objects(objects)
vis.show()

### Generate PNG frames of the animation

In [ ]:
frames = animator.create_png_sequence(
    folder="./frames_sequence",
    ease=EaseType.EASE_IN_OUT,
)

In [ ]:
print("Generated frames:", len(frames))
ipynb_display_multiple_images_in_row(
    [str(frame) for frame in frames[::1]],
    row_size=6,
)

#### Optional MP4 export (requires `imageio-ffmpeg` or `imageio[ffmpeg]` for video encoding)

If you have dependency management set up, you can run this instead of the PNG sequence export to directly create an MP4 video of the animation. This is a convenient option if you don't need the individual frames and want a ready-to-use video file. But the PNG files are still generated as an intermediate step in the process, so you can access them if needed. They are not deleted after MP4 export, so you can keep them for further use or reference. But when rendering new animations, the PNG frames will be overwritten in the same folder, so make sure to move or rename them if you want to keep them.
>
> **Note**: Creating PNG frames works without a video encoder. For MP4 export, the optional dependency `imageio-ffmpeg` or installation of `imageio[ffmpeg]` is required. Not installed by default to avoid adding a heavy dependency for users who only want to export PNG frames.

> **Note** You may need to restart the kernel after installing `imageio-ffmpeg` for it to be recognized in the environment.

In [ ]:
mp4_path = animator.animate_to_mp4(
    output_path="../../examples/nice_camera_move.mp4",
    ease=EaseType.EASE_IN_OUT,
)

print(mp4_path.resolve())